In [1]:
import numpy as np
import pysindy as ps
import sympy as sym

In [2]:
# Define variable-specific functions
position_functions = [
    lambda x: x,           # linear
    lambda x: x**2,        # quadratic
    lambda x: x**3         # cubic
]

velocity_functions = [
    lambda x: x,           # linear
    lambda x: np.sin(x),   # sinusoidal
    lambda x: np.cos(x)    # cosinusoidal
]

acceleration_functions = [
    lambda x: x,           # linear
    lambda x: np.exp(-x),  # exponential decay
    lambda x: np.log(np.abs(x) + 1)  # logarithmic
]

In [3]:
# Create variable-specific library
position_library = ps.CustomLibrary(library_functions=position_functions,
                                    function_names=[lambda x: x, lambda x: x + '^2', lambda x: x + '^3'])
velocity_library = ps.CustomLibrary(library_functions=velocity_functions,
                                    function_names=[lambda x: x, lambda x: 'sin(' + x + ')', lambda x: 'cos(' + x + ')'])
acceleration_library = ps.CustomLibrary(library_functions=acceleration_functions,
                                         function_names=[lambda x: x, lambda x: 'e^(-' + x + ')', lambda x: 'log(|' + x + '|+1)'])
gen_library = ps.GeneralizedLibrary([position_library, velocity_library, acceleration_library])

# Generate sample data
t = np.linspace(0, 10, 1000)
x = np.column_stack([
    np.sin(t),           # position-like variable
    np.cos(t),           # velocity-like variable  
    -np.sin(t)           # acceleration-like variable
])

print(x.shape)
print(t.shape)

(1000, 3)
(1000,)


In [4]:
# Fit SINDy model
out = gen_library.fit(x)
out = gen_library.transform(x)
print(out.shape)
print(out)

# Print the equations from the SINDy fit
model = ps.SINDy(feature_library=gen_library, optimizer=ps.SSR(alpha=0.00001)) # feature_names=["x", "x\'", "x\'\'"], 
model.fit(x, t=t)
model.print(precision=5)

(1000, 27)
[[ 0.          1.         -0.         ...  0.          0.69314718
   0.        ]
 [ 0.01000984  0.9999499  -0.01000984 ...  0.00996008  0.69312213
   0.00996008]
 [ 0.02001868  0.99979961 -0.02001868 ...  0.01982094  0.69304698
   0.01982094]
 ...
 [-0.52711499 -0.84979397  0.52711499 ...  0.42338033  0.61507427
   0.42338033]
 [-0.53559488 -0.84447506  0.53559488 ...  0.42891785  0.61219472
   0.42891785]
 [-0.54402111 -0.83907153  0.54402111 ...  0.43439012  0.60926084
   0.43439012]]
(x0)' = 0.33332 x1 + 0.00001 x1^3 + 0.33332 x1 + 0.00003 sin(x1) + 0.33332 x1
(x1)' = -0.16666 x0 + 0.16666 x2 + -0.16666 x0 + 0.16666 x2 + -0.16666 x0 + 0.16666 x2
(x2)' = -0.33332 x1 + -0.00001 x1^3 + -0.33332 x1 + -0.00003 sin(x1) + -0.33332 x1


In [5]:
a = sym.Rational(1,2) + sym.Rational(1,3)
print(a)
a.evalf()

5/6


0.833333333333333